# Partie 1 : CIFAR

In [ ]:
import tensorflow as tf
import numpy as np

# 1. Chargement du dataset CIFAR-10
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
(x_train_no_norm, y_train), (x_test_no_norm, y_test) = tf.keras.datasets.cifar10.load_data()

x_train_norm, x_test = x_train / 255.0, x_test / 255.0

moyennes = np.mean(x_train_norm, axis=(0, 1, 2))
ecarts_types = np.std(x_train_norm, axis=(0, 1, 2))

print("Moyennes (R, G, B) :", moyennes)
print("Écarts-types (R, G, B) :", ecarts_types)

# Affichage des dimensions pour vérifier
print("Format des données d'entraînement :", x_train.shape)
print("Format des étiquettes d'entraînement :", y_train.shape)

## Analyse exploratoire des données

### Distribution des classes

In [ ]:
import numpy as np

import matplotlib.pyplot as plt
import numpy as np

# On définit les noms des classes pour que le graphique soit lisible
noms_classes = ['Avion', 'Automobile', 'Oiseau', 'Chat', 'Cerf', 
                'Chien', 'Grenouille', 'Cheval', 'Bateau', 'Camion']

# On compte le nombre d'occurrences de chaque classe dans y_train
# y_train contient les étiquettes de 0 à 9. np.unique compte combien de fois chaque chiffre apparaît.
classes, nombres = np.unique(y_train, return_counts=True)

# Création du graphique en barres
plt.figure(figsize=(10, 6))
plt.bar(noms_classes, nombres, color='skyblue', edgecolor='black')

# Personnalisation (titre, légendes)
plt.title("Distribution des classes dans CIFAR-10 (Données d'entraînement)", fontsize=14)
plt.xlabel("Classes", fontsize=12)
plt.ylabel("Nombre d'images", fontsize=12)
plt.xticks(rotation=45) # On incline le texte pour éviter que les mots se chevauchent

# On ajoute le nombre exact au-dessus de chaque barre pour plus de clarté
for i in range(len(classes)):
    plt.text(i, nombres[i] + 100, str(nombres[i]), ha='center')

# Affichage
plt.tight_layout()
plt.show()

### Affichage des 10 premières images

In [ ]:
plt.figure(figsize=(12, 5))

# On affiche les 10 premières images du dataset
for i in range(10):
    plt.subplot(2, 5, i + 1)
    # x_train[i] est déjà normalisé entre 0 et 1, imshow l'affiche sans problème
    plt.imshow(x_train[i]) 
    
    # On récupère le nom de la classe correspondante
    index_classe = int(y_train[i][0])
    plt.title(noms_classes[index_classe])
    
    plt.axis('off') # On cache les axes (les numéros de pixels) pour faire plus joli

plt.tight_layout()
plt.show()

## Algortihme de ML 

### XGBoost

In [ ]:
x_train_xgb = x_train_norm.reshape(x_train_norm.shape[0], -1)
x_test_xgb = x_test.reshape(10000,-1)

In [ ]:
from sklearn.model_selection import GridSearchCV
import xgboost as xgb

model = xgb.XGBClassifier(device='cuda')

param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.1, 0.01],
    'n_estimators': [100, 200]
}

grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5)
grid_search.fit(x_train_xgb, y_train)



print(f"Meilleurs paramètres : {grid_search.best_params_}")

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split



model_xgb = xgb.XGBClassifier(learning_rate=0.1, max_depth=7, n_estimators=200, device='cuda')
model_xgb.fit(x_train_xgb,y_train.ravel())

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
y_pred_xgb = model_xgb.predict(x_test_xgb)
print(f'Accuracy : {accuracy_score(y_test,y_pred_xgb)}')
print(f'report : {classification_report(y_test, y_pred_xgb)}')

class_names = ['Avion', 'Automobile', 'Oiseau', 'Chat', 'Cerf', 'Chien', 'Grenouille', 'Cheval', 'Bateau', 'Camion']

# 2. Calculer la matrice
cm = confusion_matrix(y_test, y_pred_xgb)

# 3. Afficher la matrice avec les noms
fig, ax = plt.subplots(figsize=(8, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=noms_classes)

# Tracer avec une palette de couleurs (cmap)
disp.plot(cmap=plt.cm.Blues, ax=ax, xticks_rotation='vertical')

plt.title("Matrice de Confusion")
plt.show()

In [ ]:
plt.imshow(x_test[1].reshape(32, 32, 3)) # Remettre en forme pour l'affichage
plt.title(f"Vrai: {y_test[1]}, Prédit: {y_pred_xgb[1]}")
plt.show()

## RandomForest avec PCA

In [ ]:
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

x_train_flat = x_train_norm.reshape(x_train_norm.shape[0], -1)
# On crée une chaîne de traitement (Pipeline)
model_rf_pca = Pipeline([
    ('compression_pca', PCA(n_components=100)), # Étape 1 : Réduire à 100 composantes
    ('classifieur_rf', RandomForestClassifier()) # Étape 2 : Le Random Forest
])

param_grid = {
    'classifieur_rf__max_depth': [10, 15, 20], 
    'classifieur_rf__n_estimators': [100, 200]
}

grid_search = GridSearchCV(estimator=model_rf_pca, param_grid=param_grid, cv=5)
grid_search.fit(x_train_flat, y_train.ravel())


print(f"Meilleurs paramètres : {grid_search.best_params_}")

In [ ]:
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline



model_rf_pca = Pipeline([
    ('compression_pca', PCA(n_components=100)), 
    ('classifieur_rf', RandomForestClassifier(max_depth=20, n_estimators=200, random_state=42)) 
])


model_rf_pca.fit(x_train_flat, y_train.ravel())

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns


x_test_flat = x_test.reshape(x_test.shape[0], -1) 


print("Génération des prédictions en cours...")
y_pred = model_rf_pca.predict(x_test_flat)


accuracy = accuracy_score(y_test, y_pred)
print(f"\nExactitude (Accuracy) sur le jeu de test : {accuracy * 100:.2f}%")
print("\nRapport de classification détaillé :")
print(classification_report(y_test, y_pred))


cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Matrice de Confusion')
plt.xlabel('Classes Prédites (Ce que le modèle répond)')
plt.ylabel('Classes Réelles (La vérité)')
plt.show()

# CNN classique

## Optimisation avec Keras Tuner

In [ ]:
import tensorflow as tf
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# 1. Définition de la fonction de construction du modèle
def build_model(hp):
    model = Sequential()
    
    # --- PREMIÈRE COUCHE CONVOLUTIVE ---
    hp_filters_1 = hp.Int('conv_1_filters', min_value=32, max_value=128, step=32) # choix du nombre de filtres
    model.add(Conv2D(filters=hp_filters_1, kernel_size=(3, 3), activation='relu', input_shape=(32, 32, 3)))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    
    # --- DEUXIÈME COUCHE CONVOLUTIVE ---
    hp_filters_2 = hp.Int('conv_2_filters', min_value=64, max_value=256, step=64)
    model.add(Conv2D(filters=hp_filters_2, kernel_size=(3, 3), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    
    # --- TROISIÈME COUCHE CONVOLUTIVE ---
    hp_filters_3 = hp.Int('conv_3_filters', min_value=64, max_value=256, step=64)
    model.add(Conv2D(filters=hp_filters_3, kernel_size=(3, 3), activation='relu', padding='same'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    
    # --- PASSAGE EN 1D (FLATTEN) ---
    model.add(Flatten())
    
    # --- COUCHE CACHÉE (DENSE) ---
    hp_units = hp.Int('dense_units', min_value=128, max_value=512, step=128)
    model.add(Dense(units=hp_units, activation='relu'))
    
    # --- DROPOUT (Contre le surapprentissage) ---
    hp_dropout = hp.Float('dropout_rate', min_value=0.2, max_value=0.5, step=0.1)
    model.add(Dropout(rate=hp_dropout))
    
    # --- COUCHE DE SORTIE (CIFAR-10 = 10 classes) ---
    model.add(Dense(10, activation='softmax'))
    
    # --- COMPILATION ---
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=hp_learning_rate),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    
    return model

# 2. Instanciation du Tuner
tuner = kt.Hyperband(
    build_model,
    objective='val_accuracy', 
    max_epochs=10,            
    factor=3,
    directory='mon_dossier_tuner', 
    project_name='optimisation_cifar10_3couches' 
)

# 3. Création d'un callback d'arrêt anticipé
stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)

# 4. Lancement de la recherche
print("Lancement de la recherche des meilleurs hyperparamètres avec 3 couches...")
tuner.search(x_train_norm, y_train, 
             epochs=20, 
             validation_split=0.2, 
             callbacks=[stop_early])

# 5. Récupération des résultats
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print("\n--- RECHERCHE TERMINÉE ---")
print(f"Meilleur nombre de filtres (Conv 1) : {best_hps.get('conv_1_filters')}")
print(f"Meilleur nombre de filtres (Conv 2) : {best_hps.get('conv_2_filters')}")
print(f"Meilleur nombre de filtres (Conv 3) : {best_hps.get('conv_3_filters')}") # Affichage de la 3ème couche
print(f"Meilleurs neurones (Dense) : {best_hps.get('dense_units')}")
print(f"Meilleur taux de Dropout : {best_hps.get('dropout_rate')}")
print(f"Meilleur Learning Rate : {best_hps.get('learning_rate')}")

## Construction du modèle

In [ ]:
# 1. On demande au Tuner de reconstruire le modèle avec les MEILLEURS paramètres trouvés
meilleurs_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
modele_champion = tuner.hypermodel.build(meilleurs_hps)

# 2. Configuration de l'arrêt anticipé (Early Stopping) revisité
stop_early_final = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=5,                 # On est plus tolérant (on attend 5 époques sans amélioration)
    restore_best_weights=True   # CRUCIAL : à la fin, on recharge les poids de la meilleure époque, pas ceux de la dernière !
)

# 3. Lancement de l'entraînement final
print("Début de l'entraînement du modèle champion...")

# On sauvegarde l'historique pour pouvoir tracer des graphiques ensuite
historique = modele_champion.fit(
    x_train_norm, y_train,
    epochs=50,                  # On met un grand nombre d'époques, l'Early Stopping l'arrêtera au bon moment
    batch_size=64,              # Le modèle analyse les images par paquets de 64 (plus rapide et plus stable)
    validation_split=0.2,       # On garde 20% de x_train pour valider à chaque époque
    callbacks=[stop_early_final]
)

# 4. Le verdict final sur les données de TEST (x_test, y_test)
print("\n--- ÉVALUATION FINALE ---")
perte, precision = modele_champion.evaluate(x_test, y_test)
print(f"Précision finale sur les données de test inconnues : {precision * 100:.2f}%")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 5))

# --- GRAPHIQUE 1 : LA PRÉCISION (ACCURACY) ---
plt.subplot(1, 2, 1)
plt.plot(historique.history['accuracy'], label='Entraînement', color='blue', linewidth=2)
plt.plot(historique.history['val_accuracy'], label='Validation', color='orange', linewidth=2)
plt.title('Évolution de la Précision', fontsize=14)
plt.xlabel('Époques', fontsize=12)
plt.ylabel('Précision', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

# --- GRAPHIQUE 2 : L'ERREUR (LOSS) ---
plt.subplot(1, 2, 2)
plt.plot(historique.history['loss'], label='Entraînement', color='blue', linewidth=2)
plt.plot(historique.history['val_loss'], label='Validation', color='orange', linewidth=2)
plt.title('Évolution de l\'Erreur', fontsize=14)
plt.xlabel('Époques', fontsize=12)
plt.ylabel('Erreur (Loss)', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

print("1. Génération des prédictions sur les données de test...")
# Le modèle sort des probabilités pour chaque classe (ex: [0.1, 0.8, 0.05, ...])
predictions_probabilites = modele_champion.predict(x_test)

# On récupère l'index de la probabilité la plus haute pour chaque image (la classe prédite)
y_pred = np.argmax(predictions_probabilites, axis=1)

# On s'assure que y_test est bien un vecteur 1D plat (parfois CIFAR a une forme (10000, 1))
y_test_plat = y_test.flatten() if y_test.ndim > 1 else y_test

# Les noms des classes (à adapter si tu as changé de dataset)
classes_noms = ['Avion', 'Voiture', 'Oiseau', 'Chat', 'Cerf', 
                'Chien', 'Grenouille', 'Cheval', 'Bateau', 'Camion']

print("2. Création et affichage de la matrice de confusion...")
# Configuration de la taille de la figure
fig, ax = plt.subplots(figsize=(10, 8))

# Calcul de la matrice
cm = confusion_matrix(y_test_plat, y_pred)

# Création de l'affichage visuel
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes_noms)

# Tracé de la matrice avec une carte de couleurs bleue
disp.plot(cmap=plt.cm.Blues, ax=ax, xticks_rotation=45)

plt.title("Matrice de Confusion : Modèle Champion (Keras)", fontsize=14, pad=20)
plt.tight_layout()
plt.show()

# Finetuning - EfficientNet b4

## Optimisation

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision.transforms import v2
import optuna

val_split = 0.2
batch_size = 64
img_size = 224

x_tensor = torch.tensor(x_train_no_norm, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
y_tensor = torch.tensor(y_train, dtype=torch.long).squeeze()

full_ds = TensorDataset(x_tensor, y_tensor)
len_val = int(val_split * len(full_ds))
len_train = len(full_ds) - len_val
train_ds, val_ds = random_split(full_ds, [len_train, len_val])

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device utilisé : {device}")

train_transforms = v2.Compose([
    v2.ToDtype(torch.float32, scale=False),  
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomRotation(degrees=15),
    v2.ColorJitter(brightness=0.2, contrast=0.2),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = v2.Compose([
    v2.ToDtype(torch.float32, scale=False),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


def objective(trial):
    print(f"\nDémarrage du Trial #{trial.number}")

    lr            = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    dropout_rate  = trial.suggest_float("dropout_rate", 0.2, 0.6)
    weight_decay  = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)

    base_model = efficientnet_b4(weights=EfficientNet_B4_Weights.IMAGENET1K_V1)
    base_model.classifier = nn.Sequential(
        nn.Dropout(p=dropout_rate),
        nn.Linear(in_features=1792, out_features=10)
    )

    model = nn.Sequential(
        nn.Upsample(size=(img_size, img_size), mode='bilinear', align_corners=False),
        base_model
    ).to(device)

    for param in model.parameters():
        param.requires_grad = False
    for param in model[1].features[-1].parameters():
        param.requires_grad = True
    for param in model[1].classifier.parameters():
        param.requires_grad = True

    model.eval()
    model[1].classifier.train()

    parametres_a_entrainer = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.Adam(parametres_a_entrainer, lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    epochs = 5

    for epoch in range(epochs):
        model[1].classifier.train()

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            inputs = train_transforms(inputs)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        model.eval()
        correct = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)

                # application des val_transforms (normalisation)
                inputs = val_transforms(inputs)

                outputs = model(inputs)
                correct += (outputs.argmax(1) == labels).sum().item()

        val_acc = correct / len(val_ds)

        trial.report(val_acc, epoch)
        if trial.should_prune():
            del model
            torch.cuda.empty_cache()
            raise optuna.exceptions.TrialPruned()

    del model
    torch.cuda.empty_cache()
    return val_acc


if __name__ == "__main__":
    print("\nDémarrage de l'optimisation")
    study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner())
    study.optimize(objective, n_trials=20)

    print("\nOPTIMISATION TERMINÉE")
    print("Meilleur score de validation :", study.best_value)
    print("Meilleurs Hyperparamètres trouvés :")
    for key, value in study.best_params.items():
        print(f"  - {key} : {value}")

## Phase 1

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision.transforms import v2

val_split = 0.2
batch_size = 64

x_tensor = torch.tensor(x_train_no_norm, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
y_tensor = torch.tensor(y_train, dtype=torch.long).squeeze()

full_ds = TensorDataset(x_tensor, y_tensor)
len_val = int(val_split * len(full_ds))
len_train = len(full_ds) - len_val
train_ds, val_ds = random_split(full_ds, [len_train, len_val])

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")


base_model = efficientnet_b4(weights=EfficientNet_B4_Weights.IMAGENET1K_V1)
base_model.classifier = nn.Sequential(
    nn.Dropout(p=0.26269347283865074),
    nn.Linear(in_features=1792, out_features=10)
)

for param in base_model.parameters():
    param.requires_grad = False
for param in base_model.features[-1].parameters():
    param.requires_grad = True
for param in base_model.classifier.parameters():
    param.requires_grad = True

base_model = base_model.to(device)


base_model.eval()
base_model.features[-1].train()
base_model.classifier.train()

criterion = nn.CrossEntropyLoss()
parametres_a_entrainer = [p for p in base_model.parameters() if p.requires_grad]
optimizer = optim.Adam(parametres_a_entrainer, lr=0.004336516134836935, weight_decay=1.7086691815731318e-05)


train_transforms = v2.Compose([
    v2.Resize((224, 224), interpolation=v2.InterpolationMode.BILINEAR),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomRotation(degrees=15),
    v2.ColorJitter(brightness=0.2, contrast=0.2),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = v2.Compose([
    v2.Resize((224, 224), interpolation=v2.InterpolationMode.BILINEAR),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("\nLANCEMENT DE L'ENTRAÎNEMENT")

best_loss = float('inf')
patience, patience_counter = 5, 0  

for epoch in range(10):
    base_model.eval()
    base_model.features[-1].train()
    base_model.classifier.train()

    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        inputs = train_transforms(inputs)

        optimizer.zero_grad()
        outputs = base_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    base_model.eval()
    val_loss = 0.0
    correct = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            inputs = val_transforms(inputs)
            outputs = base_model(inputs)
            val_loss += criterion(outputs, labels).item()
            correct += (outputs.argmax(1) == labels).sum().item()

    val_loss /= len(val_loader)
    val_acc = correct / len(val_ds)
    print(f"Époque {epoch+1}/10 - Loss train: {running_loss/len(train_loader):.4f} "
          f"- Loss val: {val_loss:.4f} - Acc val: {val_acc:.4f}")

    if val_loss < best_loss:
        best_loss = val_loss
        patience_counter = 0
        torch.save(base_model.state_dict(), 'efficientnet_phase1.pth')
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping !")
            break

print("\nPhase terminée ! Meilleurs poids sauvegardés.")

## Phase 2

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision.transforms import v2

val_split = 0.2
batch_size = 16

x_tensor = torch.tensor(x_train_no_norm, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
y_tensor = torch.tensor(y_train, dtype=torch.long).squeeze()

full_ds = TensorDataset(x_tensor, y_tensor)
len_val = int(val_split * len(full_ds))
len_train = len(full_ds) - len_val
train_ds, val_ds = random_split(full_ds, [len_train, len_val])

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")

base_model = efficientnet_b4(weights=None)
base_model.classifier = nn.Sequential(
    nn.Dropout(p=0.26269347283865074),
    nn.Linear(in_features=1792, out_features=10)
)

base_model = base_model.to(device)
base_model.load_state_dict(torch.load('efficientnet_phase1.pth'))
print("Poids de la Phase 1 chargés.")

for param in base_model.parameters():
    param.requires_grad = True
for module in base_model.modules():
    if isinstance(module, nn.BatchNorm2d):
        module.eval()
        for param in module.parameters():
            param.requires_grad = False

optimizer = optim.Adam([
    {'params': base_model.features[:4].parameters(), 'lr': 1e-6},
    {'params': base_model.features[4:].parameters(), 'lr': 5e-6},
    {'params': base_model.classifier.parameters(),   'lr': 1e-5},
], weight_decay=1.7086691815731318e-05)

epochs = 15
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=epochs, eta_min=1e-7
)

criterion = nn.CrossEntropyLoss()

train_transforms = v2.Compose([
    v2.Resize((224, 224), interpolation=v2.InterpolationMode.BILINEAR),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomRotation(degrees=15),
    v2.ColorJitter(brightness=0.2, contrast=0.2),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = v2.Compose([
    v2.Resize((224, 224), interpolation=v2.InterpolationMode.BILINEAR),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

best_loss = float('inf')
patience, patience_counter = 4, 0

print("\nLANCEMENT DE L'ENTRAÎNEMENT")

for epoch in range(epochs):
    base_model.train()
    for module in base_model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()

    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        inputs = train_transforms(inputs)

        optimizer.zero_grad()
        outputs = base_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    base_model.eval()
    val_loss = 0.0
    correct = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            inputs = val_transforms(inputs)
            outputs = base_model(inputs)
            val_loss += criterion(outputs, labels).item()
            correct += (outputs.argmax(1) == labels).sum().item()

    val_loss /= len(val_loader)
    val_acc = correct / len(val_ds)
    current_lr = scheduler.get_last_lr()[0]
    print(f"Époque {epoch+1}/{epochs} - Loss train: {running_loss/len(train_loader):.4f} "
          f"- Loss val: {val_loss:.4f} - Acc val: {val_acc:.4f} - LR: {current_lr:.2e}")

    scheduler.step()

    if val_loss < best_loss:
        best_loss = val_loss
        patience_counter = 0
        torch.save(base_model.state_dict(), 'efficientnet_final.pth')
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping déclenché.")
            break

print("\nTerminé. Modèle sauvegardé.")

## Test

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b4
from torchvision.transforms import v2
from torch.utils.data import DataLoader, TensorDataset

print("PRÉPARATION DES DONNÉES DE TEST")
x_test_tensor = torch.tensor(x_test_no_norm, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
y_test_tensor = torch.tensor(y_test, dtype=torch.long).squeeze()

test_ds = TensorDataset(x_test_tensor, y_test_tensor)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=0, pin_memory=True)

print("=== RECONSTRUCTION ET CHARGEMENT DU MODÈLE ===")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

base_model = efficientnet_b4(weights=None)
base_model.classifier = nn.Sequential(    
    nn.Dropout(p=0.26269347283865074),
    nn.Linear(in_features=1792, out_features=10)
)

model = nn.Sequential(
    nn.Upsample(size=(224, 224), mode='bilinear', align_corners=False),
    base_model
).to(device)

model[1].load_state_dict(torch.load('efficientnet_final.pth', map_location=device))
print("Poids rechargés avec succès !")

# FIX : normalisation du jeu de test
test_transforms = v2.Compose([
    v2.ToDtype(torch.float32, scale=False),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("\n=== LANCEMENT DES PRÉDICTIONS ===")
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        inputs = test_transforms(inputs)  # FIX : normalisation manquante
        outputs = model(inputs)
        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total
print(f"\nRÉSULTAT FINAL : Précision sur l'ensemble de test = {accuracy * 100:.2f}%")

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b4
from torchvision.transforms import v2
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Noms des classes CIFAR-10
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

# ── 1. Reconstruction du modèle et chargement des poids ─────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

base_model = efficientnet_b4(weights=None)
base_model.classifier = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(in_features=1792, out_features=10)
)
model = nn.Sequential(
    nn.Upsample(size=(224, 224), mode='bilinear', align_corners=False),
    base_model
).to(device)

model[1].load_state_dict(torch.load('efficientnet_final.pth', map_location=device))
print("Poids rechargés depuis efficientnet_final.pth !")

# ── 2. Préparation des données de test ──────────────────────────────────────
x_test_tensor = torch.tensor(x_test_no_norm, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
y_test_tensor = torch.tensor(y_test, dtype=torch.long).squeeze()

test_loader = DataLoader(TensorDataset(x_test_tensor, y_test_tensor),
                         batch_size=64, shuffle=False, num_workers=0, pin_memory=True)

test_transforms = v2.Compose([
    v2.ToDtype(torch.float32, scale=False),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ── 3. Génération des prédictions ────────────────────────────────────────────
print("Génération des prédictions en cours...")
model.eval()
all_preds  = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs  = test_transforms(inputs.to(device))
        outputs = model(inputs)
        all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.numpy())

y_pred = np.array(all_preds)
y_true = np.array(all_labels)

# ── 4. Métriques ─────────────────────────────────────────────────────────────
accuracy = accuracy_score(y_true, y_pred)
print(f"\nExactitude (Accuracy) sur le jeu de test : {accuracy * 100:.2f}%")
print("\nRapport de classification détaillé :")
print(classification_report(y_true, y_pred, target_names=class_names))

# ── 5. Matrice de confusion ───────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Matrice de Confusion — EfficientNet B4')
plt.xlabel('Classes Prédites (Ce que le modèle répond)')
plt.ylabel('Classes Réelles (La vérité)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Resnet-20

Resnet-20 est un réseau de neurones résiduel. En effet, il permet de faire des sauts entre les couches. Cela a pour conséquence d'éviter la perte du gradient. Aussi, en sautant des couches, il est possible de sauter la mise à jour de certains poids et donc saute des couches inutiles. Resnet-20 a justement été conçu pour le dataset CIRFAR-10 et possède par conséquent 20 couches.

Les couches ont toutes, de manière classique, une connexion vers la couche suivante. Néanmoins, les connexions résiduelles sont définies toutes les deux couches. Autrement dit, le résultat de la couche de départ va fusionner avec la sortie de la couche 3 sans passer par la couche 1 et 2. De manière évidente, la sortie de la couche d'entré est additionner avec la sortie de la convolution/couche 2 **avant** de passer par l'activation. Il y a donc dans Resnet-20 9 connexions résiduelles. Sachant que chaque bloque résiduel est constitué de 2 couches, il y a 18 couches de convolution + l'entrée + la couche dense de sortie qui donne bien 20 couches au total.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("xpu" if torch.xpu.is_available() else "cpu")
print(f"Lancement de l'entraînement sur : {device}")

# A. Le moule du bloc résiduel
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()
        
        # Le chemin complexe (2 convolutions)
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # Le raccourci (Le fameux adaptateur 1x1 si la dimension change)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        shortcut = self.shortcut(x)          # L'information d'origine sécurisée
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += shortcut                      # LA FUSION
        out = F.relu(out)
        return out

# B. L'assemblage du ResNet-20
class ResNet20(nn.Module):
    def __init__(self, num_classes=10):
        super(ResNet20, self).__init__()
        self.in_channels = 16
        
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        
        # Les 9 connexions résiduelles réparties en 3 groupes
        self.layer1 = self._make_layer(16, num_blocks=3, stride=1)
        self.layer2 = self._make_layer(32, num_blocks=3, stride=2)
        self.layer3 = self._make_layer(64, num_blocks=3, stride=2)
        
        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(64, num_classes)

    def _make_layer(self, out_channels, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(ResidualBlock(self.in_channels, out_channels, s))
            self.in_channels = out_channels
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.avg_pool(out)
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        return out

print("Préparation des données CIFAR-10...")

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    # Normalisation
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

# On utilise les 50 000 images pour l'entraînement
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
train_loader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)

# On valide sur le set de test de 10 000 images
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
val_loader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=False, num_workers=2)

# 2. LES HYPERPARAMÈTRES EXACTS DU PAPIER
model = ResNet20().to(device)
criterion = nn.CrossEntropyLoss()

# SGD avec Momentum 0.9 et Weight Decay 1e-4
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)

# LE SECRET : Les cassures forcées aux époques 82 et 123 (approx 32k et 48k itérations)
scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[82, 123], gamma=0.1)


# ==========================================================
# 4. LA BOUCLE D'ENTRAÎNEMENT SÉCURISÉE (Avec .pth)
# ==========================================================
num_epochs = 160
best_val_accuracy = 0.0

print(f"\nDémarrage de l'entraînement sur {device}...")
num_epochs = 160 # On va jusqu'au bout
best_val_accuracy = 0.0

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).cpu().sum().item()
            
    val_accuracy = correct / total
    
    # Le scheduler avance indépendamment de la performance
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    
    print(f"Ép. [{epoch+1}/{num_epochs}] - LR: {current_lr:.5f} - Val_Acc: {val_accuracy*100:.2f}%")

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        torch.save(model.state_dict(), 'resnet20_best_model.pth')



print(f"Le meilleur modèle a atteint {best_val_accuracy*100:.2f}% de précision.")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

# --- 1. S'assurer que l'architecture du Resnet-20 est bien connue ---
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        shortcut = self.shortcut(x)
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += shortcut
        out = F.relu(out)
        return out

class ResNet20(nn.Module):
    def __init__(self, num_classes=10):
        super(ResNet20, self).__init__()
        self.in_channels = 16
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.layer1 = self._make_layer(16, num_blocks=3, stride=1)
        self.layer2 = self._make_layer(32, num_blocks=3, stride=2)
        self.layer3 = self._make_layer(64, num_blocks=3, stride=2)
        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(64, num_classes)

    def _make_layer(self, out_channels, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(ResidualBlock(self.in_channels, out_channels, s))
            self.in_channels = out_channels
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.avg_pool(out)
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        return out

device = torch.device("xpu" if torch.xpu.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))

# --- 2. Initialisation du modèle et chargement des poids ---
model = ResNet20(num_classes=10).to(device)
model.load_state_dict(torch.load('resnet20.pth', map_location=device))
model.eval()

# --- 3. (Re)Préparation du dataloader de TEST ---
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

# Il est important de bien utiliser le set d'évaluation CIFAR10 (train=False) comme test 
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
test_loader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=False, num_workers=0)


# --- 4. Calcul de la matrice de confusion ---
print("Génération en cours des prédictions sur le set de test...")
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

# --- 5. Calcul de l'Accuracy ---
acc = accuracy_score(all_labels, all_preds)
print(f"\nExactitude (Accuracy) sur l'ensemble de test : {acc * 100:.2f}%\n")

# --- 6. Affichage avec Matplotlib ---
class_names = ['Avion', 'Automobile', 'Oiseau', 'Chat', 'Cerf', 'Chien', 'Grenouille', 'Cheval', 'Bateau', 'Camion']

cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

fig, ax = plt.subplots(figsize=(10, 8))
disp.plot(cmap=plt.cm.Blues, ax=ax, xticks_rotation='vertical')

plt.title("Matrice de Confusion — ResNet-20 sur l'ensemble de Test")
plt.tight_layout()
plt.show()

# CNN Hybride

Au lieu de prendre un CNN pré-entrainé, nous avons choisi de reprendre efficientNet b4 car nous avions de très bons résultats. L'objectif ici est de voir si en le combinant avec un XGBoost, nous pourrions encore augmenter son accuracy.

Pour ce faire, nous rechargeons les poids appris et nous supprimons la dernière couche ou plus exactement en la remplaçant par une couche d'identité. Ensuite, on fait passer toutes les images et on stocke les vecteurs de sorties dans un grand tableau numpy(pour que XGBoost puisse les lire) de taille 50 000 * 1792. Ensuite, on entraine notre modèle XGBoost sur ce "datatset" qui n'est autre que les caractéristiques extraites par efficientNet b4. 

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import TensorDataset, DataLoader, random_split
from torchvision.models import efficientnet_b4
import xgboost as xgb
import numpy as np
import os
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Lancement de l'entraînement sur : {device}")

x_tensor = torch.tensor(x_train_no_norm, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
y_tensor = torch.tensor(y_train, dtype=torch.long).squeeze()

full_ds = TensorDataset(x_tensor, y_tensor)

# Split Entraînement / Validation
val_split = 0.2
len_val = int(val_split * len(full_ds))
len_train = len(full_ds) - len_val
train_ds, val_ds = random_split(full_ds, [len_train, len_val])

batch_size = 64 # Taille de batch raisonnable pour ne pas saturer la VRAM
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = efficientnet_b4(num_classes=10)

# Chargement des poids 
chemin_poids = 'efficientnet_final.pth'
model.load_state_dict(torch.load(chemin_poids, map_location=device), strict=False)

# On met un dernière couche qui ne fait rien = supprimer la dernière couche
model.classifier = nn.Identity()
model = model.to(device)
model.eval() # Mode évaluation : fige les poids et désactive le dropout



# Ce transformateur sera appliqué juste avant de donner l'image au GPU/CPU
efficientnet_transform = T.Compose([
    T.Resize((224, 224), antialias=True),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


def extract_features(dataloader, desc="Extraction"):
    features_list = []
    labels_list = []
    
    with torch.no_grad(): # Bloque le calcul des gradients
        for inputs, labels in tqdm(dataloader, desc=desc):
            
            # 1. On redimensionne le batch de 32x32 vers 380x380
            inputs_resized = efficientnet_transform(inputs)
            
            # 2. On envoie sur le GPU
            inputs_resized = inputs_resized.to(device)
            
            # 3. On extrait les caractéristiques
            outputs = model(inputs_resized)
            
            # 4. On stocke sur le CPU en format Numpy pour XGBoost
            features_list.append(outputs.cpu().numpy())
            labels_list.append(labels.numpy())
            
    return np.vstack(features_list), np.concatenate(labels_list)


X_train_features, y_train_features = extract_features(train_loader, desc="Extraction (Train)")
X_val_features, y_val_features = extract_features(val_loader, desc="Extraction (Validation)")

Nous avons décidé de faire un optuna dans l'optique de maximiser l'accuracy en sortie. 

In [ ]:
# Recherche des meilleurs paramètres de XGBoost avec Optuna
import optuna
from sklearn.metrics import accuracy_score, classification_report
import time

def objective(trial):
    param = {
        'objective': 'multi:softmax',
        'num_class': 10,
        'tree_method': 'hist', 
        'random_state': 42,
        'n_jobs': -1,
        # Paramètres à optimiser dynamiquement par Optuna
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0)
    }

    model = xgb.XGBClassifier(**param)
    model.fit(X_train_features, y_train_features)
    preds = model.predict(X_val_features)
    accuracy = accuracy_score(y_val_features, preds)

    return accuracy
debut=time.time()
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50) # Ajuste ce nombre selon ton temps disponible

temps_total = time.time() - debut

best_params = study.best_params
best_params['objective'] = 'multi:softmax'
best_params['num_class'] = 10
best_params['tree_method'] = 'hist'
best_params['random_state'] = 42
best_params['n_jobs'] = -1

# Instanciation du modèle de production
final_xgb_model = xgb.XGBClassifier(**best_params)

# Entraînement final (très rapide car on a déjà les meilleurs réglages)
final_xgb_model.fit(X_train_features, y_train_features)

#Evaluation final
print("\nÉvaluation du modèle hybride sur le set de validation...")
final_preds = final_xgb_model.predict(X_val_features)
final_accuracy = accuracy_score(y_val_features, final_preds)

print(f"Exactitude (Accuracy) Finale : {final_accuracy * 100:.2f}%\n")
print("Rapport de classification détaillé :")
print(classification_report(y_val_features, final_preds))

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b4
from torchvision.transforms import v2
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# ==========================================
# 1. LE CNN : EXTRACTION DES CARACTÉRISTIQUES (PYTORCH)
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Reconstruction du modèle de base (comme dans ton notebook)
base_model = efficientnet_b4(weights=None)
base_model.classifier = nn.Sequential( 
    nn.Dropout(p=0.26269347283865074),
    nn.Linear(in_features=1792, out_features=10)
)

model = nn.Sequential(
    nn.Upsample(size=(224, 224), mode='bilinear', align_corners=False),
    base_model
).to(device)

# Chargement des poids entraînés lors de ta Phase 2
model[1].load_state_dict(torch.load('efficientnet_final.pth', map_location=device))

# CRUCIAL : On remplace la dernière couche de classification par une fonction Identité 
# pour récupérer les vecteurs de 1792 dimensions au lieu des 10 probabilités
model[1].classifier = nn.Identity()
model.eval()

# Transformations pour l'inférence (doivent être identiques à celles du test)
extract_transforms = v2.Compose([
    v2.ToDtype(torch.float32, scale=False),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Fonction d'extraction 
# (À utiliser si tes variables X_train_features et X_val_features ne sont pas déjà générées)
def extract_features(loader):
    features_list = []
    labels_list = []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            inputs = extract_transforms(inputs)
            outputs = model(inputs)
            features_list.append(outputs.cpu().numpy())
            # S'assurer que les labels sont au bon format pour XGBoost
            if labels.ndim > 1:
                labels = labels.squeeze()
            labels_list.append(labels.cpu().numpy())
    return np.vstack(features_list), np.concatenate(labels_list)

# SI LES FEATURES NE SONT PAS ENCORE EXTRAITES, DÉCOMMENTE CES LIGNES :
# print("Extraction des features via EfficientNet (cela peut prendre quelques minutes)...")
# X_train_features, y_train_features = extract_features(train_loader)
# X_val_features, y_val_features = extract_features(val_loader)


# ==========================================
# 2. XGBOOST : ENTRAÎNEMENT AVEC LES MEILLEURS HYPERPARAMÈTRES
# ==========================================
# Les hyperparamètres optimaux issus de ton Trial 11
best_params = {
    'objective': 'multi:softmax',
    'num_class': 10,
    'tree_method': 'hist',
    'device': 'cuda', # Accélération GPU pour l'entraînement
    'random_state': 42,
    'n_jobs': -1,
    'max_depth': 7,
    'learning_rate': 0.29176335484062926,
    'n_estimators': 142,
    'subsample': 0.6364425904349618
}

print("\nInstanciation du modèle XGBoost de production...")
final_xgb_model = xgb.XGBClassifier(**best_params)

print("Entraînement de XGBoost sur les features extraites...")
final_xgb_model.fit(X_train_features, y_train_features)


# ==========================================
# 3. ÉVALUATION FINALE
# ==========================================
print("\nÉvaluation du modèle hybride (EfficientNet + XGBoost) sur le set de validation...")
final_preds = final_xgb_model.predict(X_val_features)
final_accuracy = accuracy_score(y_val_features, final_preds)

print(f"Exactitude (Accuracy) Finale : {final_accuracy * 100:.2f}%\n")
print("Rapport de classification détaillé :")
print(classification_report(y_val_features, final_preds))

In [ ]:
import torch.nn as nn

# ==========================================
# ÉVALUATION SUR L'ENSEMBLE DE TEST (CORRIGÉ)
# ==========================================

# 1. SÉCURITÉ : On s'assure que le CNN recrache 1792 features et non 10 classes
model[1].classifier = nn.Identity()
model.eval()

print("1. Extraction des caractéristiques du set de TEST via EfficientNet...")
# On relance l'extraction (cette fois on aura bien 1792 colonnes)
X_test_features, y_test_features = extract_features(test_loader)

# Vérification rapide avant de passer à XGBoost
print(f"Format des features de test : {X_test_features.shape}") # Doit afficher (N, 1792)

print("2. Prédictions via XGBoost...")
# On utilise le XGBoost fraîchement entraîné pour prédire les classes
test_preds = final_xgb_model.predict(X_test_features)

# 3. Affichage des résultats finaux
test_accuracy = accuracy_score(y_test_features, test_preds)

print(f"\n🏆 Exactitude (Accuracy) finale sur le set de TEST : {test_accuracy * 100:.2f}%\n")
print("Rapport de classification détaillé (Test) :")
print(classification_report(y_test_features, test_preds))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

# Les 10 classes de CIFAR-10 dans l'ordre officiel
classes_cifar10 = ['Avion', 'Voiture', 'Oiseau', 'Chat', 'Cerf', 
                   'Chien', 'Grenouille', 'Cheval', 'Bateau', 'Camion']

# Configuration de la taille de la figure pour qu'elle soit bien lisible
fig, ax = plt.subplots(figsize=(10, 8))

# Génération et affichage de la matrice
disp = ConfusionMatrixDisplay.from_predictions(
    y_test_features,  # Les vrais labels de ton set de test
    test_preds,       # Les prédictions de ton XGBoost
    display_labels=classes_cifar10,
    cmap=plt.cm.Blues, # Une jolie palette de couleurs bleue
    xticks_rotation=45, # Rotation des labels pour la lisibilité
    ax=ax
)

plt.title("Matrice de Confusion : EfficientNet-B4 + XGBoost", fontsize=14, pad=20)
plt.tight_layout()
plt.show()